# Lab 6 — Inner Products, Norms, and Gram–Schmidt
**Linear Algebra · UATX**  
*Week 7 · Prerequisites: §2.1 / §3.1 (pages 24–29)*

An **inner product** $\langle u, v \rangle$ induces a norm $\|v\| = \sqrt{\langle v,v\rangle}$ and the Cauchy–Schwarz inequality $|\langle u,v\rangle| \leq \|u\|\|v\|$.

**Gram–Schmidt** turns any basis into an orthonormal one, producing the $QR$ decomposition $A = QR$.

**Tasks**
1. Implement Gram–Schmidt; verify $A = QR$ and $Q^\top Q = I$.
2. Apply GS to the monomial basis $\{1, x, x^2, x^3\}$ under the $L^2[-1,1]$ inner product to recover Legendre polynomials.
3. Use $QR$ to solve polynomial regression more stably than normal equations.
4. Verify Cauchy–Schwarz numerically and plot the distribution of $|\langle u,v\rangle|/(\|u\|\|v\|)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate, special
np.random.seed(42)

## §1  Classical Gram–Schmidt

Given columns $a_1, \ldots, a_k$ of $A$, Gram–Schmidt produces orthonormal vectors $q_1, \ldots, q_k$ and upper-triangular $R$ such that $A = QR$:
$$q_j = \frac{a_j - \sum_{i<j}\langle a_j, q_i\rangle q_i}{\left\|a_j - \sum_{i<j}\langle a_j, q_i\rangle q_i\right\|}$$

In [ ]:
def gram_schmidt(A):
    """
    Classical Gram-Schmidt on columns of A.
    Returns Q (orthonormal columns), R (upper triangular) such that A = Q R.
    """
    m, k = A.shape
    Q = np.zeros((m, k))
    R = np.zeros((k, k))

    for j in range(k):
        v = A[:, j].copy().astype(float)
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] < 1e-14:
            raise ValueError(f'Column {j} is linearly dependent on previous columns.')
        Q[:, j] = v / R[j, j]

    return Q, R


# Test on a random 5x3 matrix
A = np.random.randn(5, 3)
Q, R = gram_schmidt(A)

print('A = Q R?', np.allclose(A, Q @ R))
print('Q^T Q = I?', np.allclose(Q.T @ Q, np.eye(3)))
print('R upper triangular?', np.allclose(R, np.triu(R)))
print('\nSingular values of A:', np.round(np.linalg.svd(A, compute_uv=False), 4))
print('Diagonal of R:       ', np.round(np.diag(R), 4), '  (= singular values for this random A, approximately)')

In [ ]:
# Modified Gram-Schmidt (numerically more stable)
def gram_schmidt_modified(A):
    """Modified Gram-Schmidt: numerically stabler variant."""
    m, k = A.shape
    Q = A.astype(float).copy()
    R = np.zeros((k, k))
    for j in range(k):
        R[j, j] = np.linalg.norm(Q[:, j])
        Q[:, j] /= R[j, j]
        for i in range(j+1, k):
            R[j, i] = Q[:, j] @ Q[:, i]
            Q[:, i] -= R[j, i] * Q[:, j]
    return Q, R

# Compare classical vs modified on a nearly-singular matrix
eps = 1e-7
A_ill = np.array([[1, 1, 1],
                  [eps, eps, 0],
                  [eps, 0, eps]], dtype=float)
Q_cls, R_cls = gram_schmidt(A_ill)
Q_mod, R_mod = gram_schmidt_modified(A_ill)
print('Near-singular matrix: ||Q^TQ - I|| classical:', np.linalg.norm(Q_cls.T @ Q_cls - np.eye(3)))
print('Near-singular matrix: ||Q^TQ - I|| modified: ', np.linalg.norm(Q_mod.T @ Q_mod - np.eye(3)))

## §2  $L^2$ Inner Product and Legendre Polynomials

On $P_n$ over $[-1,1]$ we use $\langle f, g\rangle = \int_{-1}^{1} f(x)\,g(x)\,dx$.

Applying Gram–Schmidt to $\{1, x, x^2, x^3\}$ under this inner product produces the **Legendre polynomials** (up to normalisation).

In [ ]:
def l2_inner(f, g, a=-1, b=1):
    """L^2 inner product of functions f, g on [a, b]."""
    integrand = lambda x: f(x) * g(x)
    val, _ = integrate.quad(integrand, a, b)
    return val

def l2_gram_schmidt(fns):
    """
    Gram-Schmidt for a list of functions under the L^2[-1,1] inner product.
    Returns list of orthonormal functions.
    """
    ortho = []
    for f in fns:
        v = f  # start with the function
        components = []
        for q in ortho:
            coeff = l2_inner(v, q)
            components.append((coeff, q))
        # subtract projections
        def make_residual(f_orig, comps):
            def residual(x):
                r = f_orig(x)
                for c, q in comps:
                    r -= c * q(x)
                return r
            return residual
        v = make_residual(v, components)
        norm = np.sqrt(l2_inner(v, v))
        q_new = (lambda v_cap, n: lambda x: v_cap(x) / n)(v, norm)
        ortho.append(q_new)
    return ortho


# Monomial basis {1, x, x^2, x^3}
monomials = [
    lambda x: np.ones_like(x),
    lambda x: x,
    lambda x: x**2,
    lambda x: x**3,
]

legendre_gs = l2_gram_schmidt(monomials)

# Compare to scipy's Legendre polynomials
xs = np.linspace(-1, 1, 300)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, (q, ax) in enumerate(zip(legendre_gs, axes)):
    y_gs = q(xs)
    # scipy Legendre, normalised to unit L^2 norm: ||P_i||^2 = 2/(2i+1)
    P_i = special.legendre(i)(xs)
    norm_Pi = np.sqrt(2 / (2*i + 1))
    y_leg = P_i / norm_Pi
    # Fix sign: match sign at x=1
    if np.sign(y_gs[-1]) != np.sign(y_leg[-1]):
        y_leg = -y_leg
    ax.plot(xs, y_gs, label='GS', lw=2)
    ax.plot(xs, y_leg, '--', label=f'$P_{i}$ (scipy)', lw=1.5)
    ax.set_title(f'$q_{i}$ vs $P_{i}$')
    ax.legend(fontsize=7); ax.grid(True)
plt.suptitle('Gram–Schmidt on monomials recovers Legendre polynomials')
plt.tight_layout(); plt.show()

## §3  QR Decomposition for Regression

For polynomial regression $X\beta \approx y$, the normal equations $X^\top X \beta = X^\top y$ can be ill-conditioned. Using $X = QR$ gives $R\beta = Q^\top y$, which is numerically more stable.

In [ ]:
# Generate regression data
N, d = 80, 5
x = np.random.uniform(-3, 3, N)
y = 2 + x - 0.5*x**2 + 0.05*x**3 + 0.5*np.random.randn(N)

# Vandermonde feature matrix
X = np.column_stack([x**i for i in range(d+1)])

# Method 1: Normal equations
beta_normal = np.linalg.solve(X.T @ X, X.T @ y)

# Method 2: QR decomposition
Q, R = np.linalg.qr(X, mode='reduced')   # numpy's stable QR
beta_qr = np.linalg.solve(R, Q.T @ y)

print('Normal equations beta: ', np.round(beta_normal, 4))
print('QR beta:               ', np.round(beta_qr, 4))
print('Max difference:        ', np.max(np.abs(beta_normal - beta_qr)))

# Condition number comparison
print(f'\ncond(X^T X) = {np.linalg.cond(X.T @ X):.2e}')
print(f'cond(R)     = {np.linalg.cond(R):.2e}  (should be sqrt of above)')

# Plot fit
xplot = np.linspace(-3, 3, 300)
Xplot = np.column_stack([xplot**i for i in range(d+1)])
plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=20, alpha=0.5, label='data')
plt.plot(xplot, Xplot @ beta_qr, 'r-', lw=2, label=f'QR fit (deg {d})')
plt.legend(); plt.grid(True)
plt.title(f'Polynomial Regression via QR (degree {d})')
plt.tight_layout(); plt.show()

## §4  Cauchy–Schwarz Inequality

For any $u, v \in \mathbb{R}^n$: $|\langle u,v\rangle| \leq \|u\|\,\|v\|$, with equality iff $u \parallel v$.

The **cosine** of the angle between $u$ and $v$ is $\cos\theta = \langle u,v\rangle / (\|u\|\|v\|) \in [-1,1]$.

In [ ]:
n_dim, n_trials = 10, 2000
ratios = []
for _ in range(n_trials):
    u = np.random.randn(n_dim)
    v = np.random.randn(n_dim)
    lhs = abs(u @ v)
    rhs = np.linalg.norm(u) * np.linalg.norm(v)
    assert lhs <= rhs + 1e-12, f'Cauchy-Schwarz violated! lhs={lhs}, rhs={rhs}'
    ratios.append(lhs / rhs)

ratios = np.array(ratios)
print(f'Cauchy-Schwarz satisfied for all {n_trials} random pairs in R^{n_dim}')
print(f'Max ratio |<u,v>| / (||u|| ||v||) = {ratios.max():.6f}  (must be <= 1)')

plt.figure(figsize=(7, 4))
plt.hist(ratios, bins=50, edgecolor='white')
plt.axvline(1.0, color='red', lw=2, label='Cauchy–Schwarz bound')
plt.xlabel(r'$|\langle u,v\rangle| \,/\, (\|u\|\|v\|)$')
plt.ylabel('Count')
plt.title(f'Distribution of cosine similarity, $n={n_dim}$, {n_trials} random pairs')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# When does equality hold?  When u and v are parallel (one is a scalar multiple of the other).
u = np.array([1., 2., 3., 4.])
v_parallel = 3.7 * u
v_orthog   = np.array([-2., 1., 0., 0.])

lhs_par = abs(u @ v_parallel)
rhs_par = np.linalg.norm(u) * np.linalg.norm(v_parallel)
print(f'Parallel:    |<u,v>| = {lhs_par:.4f},  ||u|| ||v|| = {rhs_par:.4f},  equal: {np.isclose(lhs_par, rhs_par)}')

lhs_ort = abs(u @ v_orthog)
rhs_ort = np.linalg.norm(u) * np.linalg.norm(v_orthog)
print(f'Orthogonal:  |<u,v>| = {lhs_ort:.4f},  ||u|| ||v|| = {rhs_ort:.4f}')